# MIO-TCD frozen image-level split
Creates reproducible 70/15/15 train/val/test from raw `train/` only. Input is inventory plus GT metadata; original MIO `test/` is never used. The fallback is deterministic quota-aware multilabel assignment because camera/sequence metadata is unavailable.

## Config

In [ ]:
DATASET_ROOT_OVERRIDE = None
SEED = 42
RATIOS = (.70, .15, .15)
FORCE_REBUILD = False

## Imports, validation and load data

In [ ]:
from pathlib import Path
import sys, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from mio_tcd_utils import PROJECT_ROOT, resolve_dataset_root, read_annotations, assert_inventory, filter_valid_inventory, create_split, split_report, save_split, TARGET_NAMES
DATASET_ROOT = resolve_dataset_root(DATASET_ROOT_OVERRIDE)
inventory_path = PROJECT_ROOT / 'data/mio_tcd/metadata/inventory.csv'
if not inventory_path.is_file(): raise FileNotFoundError('Run 01_mio_inventory.ipynb first.')
inventory_all = pd.read_csv(inventory_path, dtype={'image_id': str})
inventory, excluded_invalid = filter_valid_inventory(inventory_all)
assert_inventory(inventory)
print(f'Eligible images: {len(inventory):,} | Excluded invalid images: {len(excluded_invalid):,}')
if not excluded_invalid.empty:
    display(excluded_invalid[['image_id', 'image_path', 'invalid_bbox_count', 'source_classes', 'valid']])
annotations = read_annotations(DATASET_ROOT)

## Processing and summary

In [ ]:
manifest = create_split(inventory, SEED, RATIOS)
image_report, bbox_report = split_report(manifest, annotations)
display(image_report); display(bbox_report)
coverage = manifest.groupby('split')[[f'has_{x}' for x in TARGET_NAMES]].sum(); display(coverage)
if (coverage == 0).any().any(): print('WARNING: a target class has no image coverage in a split.')
if manifest.image_id.duplicated().any(): raise ValueError('Image leakage detected.')

## Save outputs

In [ ]:
output = save_split(manifest, DATASET_ROOT, SEED, RATIOS, FORCE_REBUILD, excluded_invalid)
print('Saved frozen split to:', output)
print('Excluded-image audit:', output / 'excluded_invalid_images.csv')
display(manifest.head())